# Directed results analysis

Examines the outputs of `query_directions_without_proto.py`
(`results/directional_resolved_without_proto/`). Run that script first - this notebook only reads its CSVs.

Sections: load -> overview -> sanity checks -> expert-edge comparison -> cross-context agreement ->
LLM tie-broken pairs.

In [1]:
import sys
from pathlib import Path
from itertools import combinations

import pandas as pd

# config resolves its paths from its own location, so importing it works
# regardless of the notebook's working directory
sys.path.insert(0, str(Path('..').resolve()))
from config import RESULT_DIR as PIPELINE_RESULT_DIR, RESULTS_ROOT

# follows config.RUN_NAME, i.e. whichever model config currently points at.
# to analyse a different run instead, swap in that run's directory:
#   PIPELINE_RESULT_DIR = RESULTS_ROOT / 'Llama-3.3-70B-Instruct-Turbo'
RESULT_DIR = PIPELINE_RESULT_DIR / "directional_resolved_without_proto"
CONTEXTS = ["kg_llm", "llm", "rag"]
METRICS = ["plausibility", "association", "temporality"]
# column holding the boolean prediction in each resolved CSV
METRIC_COL = {
    "plausibility": "Plausibility",
    "association": "Association",
    "temporality": "Temporality",
}

In [2]:
# load whatever resolved CSVs exist; report anything missing instead of crashing
results = {}  # (context, metric) -> DataFrame
missing = []
for context in CONTEXTS:
    for metric in METRICS:
        path = RESULT_DIR / f"{context}_{metric}_without_proto_resolved.csv"
        if path.exists():
            results[(context, metric)] = pd.read_csv(path)
        else:
            missing.append(path.name)

print(f"loaded {len(results)} resolved result files")
if missing:
    print(f"missing {len(missing)} (run query_directions_without_proto.py to produce them):")
    for m in missing:
        print("  -", m)

loaded 9 resolved result files


## Overview

Edge counts per configuration, and how many of those directions came from the LLM tie-break
(`Direction_Resolved == True`) versus falling out of the undirected predictions directly.

In [3]:
overview = pd.DataFrame([
    {
        "context": context,
        "metric": metric,
        "edges": len(df),
        "tie_broken": int((df["Direction_Resolved"] == True).sum()),
        "tie_broken_pct": round(100 * (df["Direction_Resolved"] == True).mean(), 1),
    }
    for (context, metric), df in results.items()
])
overview.sort_values(["metric", "context"]).reset_index(drop=True)

,context,metric,edges,tie_broken,tie_broken_pct
0,kg_llm,association,33,11,33.3
1,llm,association,47,28,59.6
2,rag,association,64,27,42.2
3,kg_llm,plausibility,83,49,59.0
4,llm,plausibility,89,47,52.8
5,rag,plausibility,85,43,50.6
6,kg_llm,temporality,3,0,0.0
7,llm,temporality,9,0,0.0
8,rag,temporality,35,9,25.7


## Sanity checks

After resolution each output should contain **at most one direction per unordered pair** and no
duplicate rows. Anything listed here indicates a resolution bug.

In [4]:
problems = []
for key, df in results.items():
    edges = set(zip(df["Var1"], df["Var2"]))
    contradictions = sorted({tuple(sorted(e)) for e in edges if (e[1], e[0]) in edges})
    dupes = int(df.duplicated(subset=["Var1", "Var2"]).sum())
    if contradictions or dupes:
        problems.append({"config": key, "contradictions": contradictions, "duplicate_rows": dupes})

if problems:
    display(pd.DataFrame(problems))
else:
    print("OK: no bidirectional contradictions and no duplicate (Var1, Var2) rows in any output")

OK: no bidirectional contradictions and no duplicate (Var1, Var2) rows in any output


## Comparison against expert edges

Confusion-matrix metrics of each configuration's directed edge set against
`data/expert_edge_pairs.csv`: **TP / FP / FN / TN, precision, TPR (recall), FDR, FPR, F1**.
An edge only counts as a true positive if the *direction* matches the expert edge.

The negative class needs a universe: we use every directed pair the pipeline actually queried
(`data/full_cleaned.csv`, both orientations, with the Sleep rename applied) - TN is the candidate
pairs that neither the model nor the experts assert.

In [5]:
expert = pd.read_csv(Path("..").resolve() / "data/expert_edge_pairs.csv").replace("Sleep", "Sleep disturbance")
expert_edges = set(zip(expert["Var1"], expert["Var2"]))

# candidate universe: every directed pair the pipeline queried, in both orientations
# (resolution can emit the direction opposite to the queried row)
full = pd.read_csv(Path("..").resolve() / "data/full_cleaned.csv").drop(columns=["Unnamed: 0"]).replace("Sleep", "Sleep disturbance")
universe = set(zip(full["var1"], full["var2"]))
universe |= {(b, a) for a, b in universe}

print(f"{len(expert_edges)} expert directed edges, {len(universe)} candidate directed pairs")
outside = expert_edges - universe
if outside:
    print(f"note: {len(outside)} expert edge(s) outside the queried candidate set (count as FN, never TN): {sorted(outside)}")

def confusion_scores(predicted):
    predicted = predicted & universe
    tp = len(predicted & expert_edges)
    fp = len(predicted - expert_edges)
    fn = len(expert_edges - predicted)
    tn = len(universe - predicted - expert_edges)
    precision = tp / (tp + fp) if tp + fp else 0.0
    tpr = tp / (tp + fn) if tp + fn else 0.0   # recall / sensitivity
    fdr = fp / (tp + fp) if tp + fp else 0.0   # 1 - precision
    fpr = fp / (fp + tn) if fp + tn else 0.0
    f1 = 2 * precision * tpr / (precision + tpr) if precision + tpr else 0.0
    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": round(precision, 3), "TPR": round(tpr, 3),
        "FDR": round(fdr, 3), "FPR": round(fpr, 3), "F1": round(f1, 3),
    }

scores = pd.DataFrame([
    {"context": context, "metric": metric, "edges": len(df),
     **confusion_scores(set(zip(df["Var1"], df["Var2"])))}
    for (context, metric), df in results.items()
])
scores.sort_values(["metric", "F1"], ascending=[True, False]).reset_index(drop=True)

64 expert directed edges, 180 candidate directed pairs


,context,metric,edges,TP,FP,FN,TN,precision,TPR,FDR,FPR,F1
0,rag,association,64,39,25,25,91,0.609,0.609,0.391,0.216,0.609
1,llm,association,47,22,25,42,91,0.468,0.344,0.532,0.216,0.396
2,kg_llm,association,33,15,18,49,98,0.455,0.234,0.545,0.155,0.309
3,rag,plausibility,85,51,34,13,82,0.600,0.797,0.400,0.293,0.685
4,llm,plausibility,89,47,42,17,74,0.528,0.734,0.472,0.362,0.614
5,kg_llm,plausibility,83,42,41,22,75,0.506,0.656,0.494,0.353,0.571
6,rag,temporality,35,20,15,44,101,0.571,0.312,0.429,0.129,0.404
7,llm,temporality,9,5,4,59,112,0.556,0.078,0.444,0.034,0.137
8,kg_llm,temporality,3,2,1,62,115,0.667,0.031,0.333,0.009,0.060


## Agreement between contexts

For each setting/metric, Jaccard overlap of the directed edge sets produced by the three contexts
(kg_llm vs llm vs rag). Low agreement means the knowledge source is driving the directions, not the data.

In [6]:
agreement_rows = []
for metric in METRICS:
    for ctx_a, ctx_b in combinations(CONTEXTS, 2):
        key_a, key_b = (ctx_a, metric), (ctx_b, metric)
        if key_a not in results or key_b not in results:
            continue
        edges_a = set(zip(results[key_a]["Var1"], results[key_a]["Var2"]))
        edges_b = set(zip(results[key_b]["Var1"], results[key_b]["Var2"]))
        union = edges_a | edges_b
        agreement_rows.append({
            "metric": metric,
            "pair": f"{ctx_a} vs {ctx_b}",
            "common": len(edges_a & edges_b),
            "jaccard": round(len(edges_a & edges_b) / len(union), 3) if union else 1.0,
        })

pd.DataFrame(agreement_rows)

,metric,pair,common,jaccard
0,plausibility,kg_llm vs llm,64,0.593
1,plausibility,kg_llm vs rag,50,0.424
2,plausibility,llm vs rag,58,0.500
3,association,kg_llm vs llm,14,0.212
4,association,kg_llm vs rag,16,0.198
5,association,llm vs rag,20,0.220
6,temporality,kg_llm vs llm,0,0.000
7,temporality,kg_llm vs rag,1,0.027
8,temporality,llm vs rag,5,0.128


## LLM tie-broken pairs

The pairs that were predicted in both directions and needed the direction prompt to break the tie.
Frequency across configurations shows which relationships are persistently ambiguous; the reasoning
column of one configuration is shown for spot-checking (rows whose reasoning is a bare error message
are failed resolutions that fell back to alphabetical order - see the run's WARNING output).

In [7]:
tie_broken = pd.concat(
    [
        df[df["Direction_Resolved"] == True].assign(context=context, metric=metric)
        for (context, metric), df in results.items()
        if (df["Direction_Resolved"] == True).any()
    ],
    ignore_index=True,
) if results else pd.DataFrame()

if len(tie_broken):
    freq = (
        tie_broken.assign(pair=tie_broken.apply(lambda r: tuple(sorted([r["Var1"], r["Var2"]])), axis=1))
        .groupby("pair").size().sort_values(ascending=False).rename("times_tie_broken")
    )
    display(freq.to_frame())
else:
    print("no tie-broken pairs in the loaded results")

,times_tie_broken
pair,
"(Alcohol, Depression)",6
"(Anxiety, Smoking)",6
"(Anxiety, Sleep disturbance)",6
"(Anxiety, Depression)",6
"(Alcohol, Sleep disturbance)",6
"(Alcohol, Obesity)",6
"(Depression, Education)",6
"(Depression, Sleep disturbance)",6
"(Education, Smoking)",6


In [8]:
# spot-check the resolved directions and reasoning for one configuration
INSPECT = ("kg_llm", "plausibility")

if INSPECT in results:
    df = results[INSPECT]
    resolved = df[df["Direction_Resolved"] == True]
    metric_reasoning_col = f"{METRIC_COL[INSPECT[1]]} Reasoning"
    with pd.option_context("display.max_colwidth", 200):
        display(resolved[["Var1", "Var2", metric_reasoning_col]])
else:
    print(f"{INSPECT} not loaded")

,Var1,Var2,Plausibility Reasoning
34,Anxiety,Alcohol,"Reasoning Process:\nStep 1: The report provides information about the relationship between alcohol consumption and various health outcomes, including cardiovascular diseases, tuberculosis, and var..."
35,CCI,Alcohol,Reasoning Process:\nStep 1: The report on the Chronic Lower Back Pain Community in Slovakia does not provide any direct evidence of a causal relationship between alcohol consumption and comorbidit...
36,Alcohol,Catastrophizing,"Reasoning Process:\nStep 1: The report does not provide a direct causal relationship between alcohol consumption and pain catastrophizing scores. However, it does mention that alcohol consumption ..."
37,Depression,Alcohol,"Reasoning Process:\nStep 1: The report provides evidence that alcohol consumption is associated with an increased risk of various health outcomes, including cardiovascular diseases such as ischaem..."
38,Education,Alcohol,"Reasoning Process:\nStep 1: The report provides information about the relationship between alcohol consumption and health outcomes, including disability-adjusted life-years (DALYs) and specific di..."
39,Fear_avoidance,Alcohol,Reasoning Process:\nStep 1: The report on the chronic lower back pain community does not provide any direct evidence or relationship between alcohol consumption and fear avoidance beliefs.\nStep 2...
40,Obesity,Alcohol,"Reasoning Process:\nStep 1: The report does not provide a direct causal relationship between alcohol consumption and obesity. Instead, it focuses on the health outcomes associated with alcohol con..."
41,Alcohol,Sleep disturbance,"Reasoning Process:\nStep 1: The report provides information about the relationship between alcohol consumption and sleep disturbance, stating that alcohol consumption can potentially influence the..."
42,Smoking,Alcohol,"Reasoning Process:\nStep 1: The report provides information about the health risks associated with alcohol consumption, including ischaemic heart disease, intracerebral haemorrhage, and ischaemic ..."
43,CCI,Anxiety,"Reasoning Process:\nStep 1: The report presents a community focused on chronic lower back pain, where age, sex, financial level, education, smoking, comorbidity index (CCI), pain_catastrophizing_s..."
